# ETF月度估值：时间合同与来源门

估值用于中期配置；短期风险层在未来匹配实验中冻结。本卡只执行真实Qlib月末日历与来源资格检查，不生成虚构估值，不训练或回测。

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import pandas as pd
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd()
while not (root/'SPEC.md').exists(): root=root.parent
sys.path.insert(0,str(root))
card=root/'research/etf_monthly_valuation_20261016'
plan=json.loads((card/'plan.json').read_text())
review=json.loads((card/'design_review.json').read_text())
assert review['plan_sha256']==hashlib.sha256((card/'plan.json').read_bytes()).hexdigest()
assert 'APPROVE' in review['decision'].upper() or 'APPROVE' in review.get('verdict','').upper()
from src.data.monthly_valuation_dataset import monthly_decision_schedule
import qlib
from qlib.data import D
from qlib.workflow import R
assert hashlib.sha256((root/'src/data/monthly_valuation_dataset.py').read_bytes()).hexdigest()==plan['code_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
schedule=monthly_decision_schedule(root/plan['provider_uri'],as_of=plan['as_of'])
assert len(schedule) and (schedule.execution_at>schedule.signal_at).all()
assert schedule.signal_at.dt.tz_localize(None).dt.to_period('M').is_unique
calendar=D.calendar(end_time=plan['as_of'],freq='day')
# Calendar evidence only: do not calculate return targets before source gate.
schedule.tail()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2026/10/16 11:26:03 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[73693:MainThread](2026-10-16 11:26:03,346) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[73693:MainThread](2026-10-16 11:26:03,348) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73693:MainThread](2026-10-16 11:26:03,348) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[73693:MainThread](2026-10-16 11:26:03,349) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[73693:MainThread](2026-10-16 11:26:03,349) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73693:MainThread](2026-10-16 11:26:03,350) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


,signal_at,execution_at
194,2026-05-29 16:15:00-04:00,2026-06-01 09:47:00-04:00
195,2026-06-30 16:15:00-04:00,2026-07-01 09:47:00-04:00
196,2026-07-31 16:15:00-04:00,2026-08-03 09:47:00-04:00
197,2026-08-31 16:15:00-04:00,2026-09-01 09:47:00-04:00
198,2026-09-30 16:15:00-04:00,2026-10-01 09:47:00-04:00


In [2]:
source_path=root/plan['source_card']/'summary.json'
source=json.loads(source_path.read_text())
source_sha=hashlib.sha256(source_path.read_bytes()).hexdigest()
source_review_path=root/plan['source_card']/'independent_evaluation.json'
source_review=json.loads(source_review_path.read_text()) if source_review_path.exists() else {}
qualified_path=root/source['qualified_input_path'] if source.get('qualified_input_path') else None
ready=bool(source.get('historical_valuation_qualified') is True and source.get('qualified_monthly_rows',0)>0 and source_review.get('historical_valuation_qualified') is True and qualified_path and qualified_path.is_file())
if ready:
    assert source_review['recorder_id']==source['recorder_id']
    assert source_review['plan_sha256']==source['plan_sha256']
    assert hashlib.sha256(qualified_path.read_bytes()).hexdigest()==source['qualified_input_sha256']
# A source pass requires a separate frozen modelling design; never auto-fit.
decision='source_ready_model_design_pending' if ready else 'needs_data_monthly_valuation_only'
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*']}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path='inputs',plan=plan,schedule=schedule,calendar=calendar,source_summary=source,source_summary_sha256=source_sha)
    rec.save_objects(artifact_path='sources',alignment_module=(root/'src/data/monthly_valuation_dataset.py').read_text())
    summary={'experiment':plan['experiment'],'decision':decision,'new_fits':0,'new_accounts':0,'objective_achieved':False,'production_enabled':False,'source_qualified_monthly_rows':source.get('qualified_monthly_rows',0),'complete_month_decisions':len(schedule),'first_decision':str(schedule.signal_at.iloc[0]),'last_decision':str(schedule.signal_at.iloc[-1]),'valuation_features_computed':False,'return_labels_computed':False,'source_summary_sha256':source_sha,'provider_uri':str(root/plan['provider_uri']),'as_of':plan['as_of'],'region':'us','market':'SOXL/SOXX','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'qlib_docs':['https://qlib.readthedocs.io/en/latest/component/data.html','https://qlib.readthedocs.io/en/latest/component/workflow.html']}
    rec.save_objects(artifact_path='analysis',summary=summary)
    artifact_dir=Path(summary['recorder_local_dir'])/'artifacts'
    rec.log_metrics(calendar_month_decisions=len(schedule),registered_artifact_count=sum(p.is_file() for p in artifact_dir.rglob('*')))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'plan':'plan.json','provider_uri':plan['provider_uri'],'artifact_policy':policy},sort_keys=False))
artifacts=Path(summary['recorder_local_dir'])/'artifacts'
manifest={'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]}
(card/'verification_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
summary

[73693:MainThread](2026-10-16 11:26:03,406) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name etf_monthly_valuation_20261016.


[73693:MainThread](2026-10-16 11:26:03,407) INFO - qlib.workflow - [exp.py:258] - Experiment 674360532386719332 starts running ...


[73693:MainThread](2026-10-16 11:26:03,482) INFO - qlib.workflow - [recorder.py:345] - Recorder 74a79f6e67ea44e782bb49c970978a56 starts running under Experiment 674360532386719332 ...


[73693:MainThread](2026-10-16 11:26:04,668) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'experiment': 'etf_monthly_valuation_20261016',
 'decision': 'needs_data_monthly_valuation_only',
 'new_fits': 0,
 'new_accounts': 0,
 'objective_achieved': False,
 'production_enabled': False,
 'source_qualified_monthly_rows': 0,
 'complete_month_decisions': 199,
 'first_decision': '2010-03-31 16:15:00-04:00',
 'last_decision': '2026-09-30 16:15:00-04:00',
 'valuation_features_computed': False,
 'return_labels_computed': False,
 'source_summary_sha256': 'babfaf36426ef5555a6b6cf998ab409f36411e0eafda5b0973d8e64b622605ea',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl',
 'as_of': '2026-10-15',
 'region': 'us',
 'market': 'SOXL/SOXX',
 'recorder_id': '74a79f6e67ea44e782bb49c970978a56',
 'experiment_id': '674360532386719332',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/etf_monthly_valuation_20261016/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/etf_monthly_valuation_20261016/qlib_recorder/674360532386719332/74a79f6e67ea44e782bb49

In [3]:
from IPython.display import Markdown, display
conclusion=f"""# 月度估值实验准备结论

真实Qlib日历确认{summary['complete_month_decisions']}个月末决策位置，末次为{summary['last_decision']}；这不是估值历史或独立样本数。
来源卡提供{summary['source_qualified_monthly_rows']}条合格月度估值，当前状态为{summary['decision']}。
已实现严格可用时点、经济日期优先、不同经济快照计数、同口径历史分位、负盈利和未知权重保留。
合成合同测试不能证明真实历史来源合格。没有生成估值特征或收益标签，没有拟合、账户或收益指标，不能比较市场阶段绩效。

不晋升，稳定超额未实现。本卡仅交付月度时间/特征合同及来源门，不声称已完成无估值/有估值组合实验。
下一步补齐来源卡列明的历史原始财报与篮子资格，然后在读取目标收益前冻结训练区间、模型、预算和标签成熟时点。
本支线缺数据不结束历史归因、指数结构来源研究或整体目标。生产和冻结前向观察未改变。

[来源资格](../etf_valuation_source_20261016/conclusion.md) · [计划](plan.json) · [摘要](summary.json)
"""
(card/'conclusion.md').write_text(conclusion)
display(Markdown(conclusion))

# 月度估值实验准备结论

真实Qlib日历确认199个月末决策位置，末次为2026-09-30 16:15:00-04:00；这不是估值历史或独立样本数。
来源卡提供0条合格月度估值，当前状态为needs_data_monthly_valuation_only。
已实现严格可用时点、经济日期优先、不同经济快照计数、同口径历史分位、负盈利和未知权重保留。
合成合同测试不能证明真实历史来源合格。没有生成估值特征或收益标签，没有拟合、账户或收益指标，不能比较市场阶段绩效。

不晋升，稳定超额未实现。本卡仅交付月度时间/特征合同及来源门，不声称已完成无估值/有估值组合实验。
下一步补齐来源卡列明的历史原始财报与篮子资格，然后在读取目标收益前冻结训练区间、模型、预算和标签成熟时点。
本支线缺数据不结束历史归因、指数结构来源研究或整体目标。生产和冻结前向观察未改变。

[来源资格](../etf_valuation_source_20261016/conclusion.md) · [计划](plan.json) · [摘要](summary.json)
